In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Matplotlib formatting
plt.style.use("default")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.autolayout"] = True

# 1. Load Data
CSV_PATH = r"C:\Users\HP-PC\Desktop\Project_X\Data\cleaned_booking.csv"
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Cleaned CSV not found at: {CSV_PATH}")

print("--> [Step 1] Loading cleaned dataset...")
df = pd.read_csv(CSV_PATH)
df["date"] = pd.to_datetime(df["date"])

# 2. Feature Engineering
print("--> [Step 2] Engineering features...")
df["booking_hour"] = pd.to_datetime(
    df["time"], format="%H:%M:%S", errors="coerce"
).dt.hour
df["fare_per_km"] = np.where(
    (df["ride_distance"].notnull()) & (df["ride_distance"] > 0),
    df["booking_value"] / df["ride_distance"],
    np.nan,
)

# 3. Parametric & Outlier Profiling (IQR Method)
print("\n================ STATISTICAL PROFILING ================")
num_cols = [
    "booking_value",
    "ride_distance",
    "avg_vtat",
    "avg_ctat",
    "customer_rating",
]
stat_summary = []

for col in num_cols:
    valid = df[col].dropna()
    q1 = valid.quantile(0.25)
    q3 = valid.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)
    outliers = valid[(valid < lower) | (valid > upper)]

    stat_summary.append(
        {
            "Metric": col,
            "Mean": round(valid.mean(), 2),
            "Median": round(valid.median(), 2),
            "Std Dev": round(valid.std(), 2),
            "Skewness": round(valid.skew(), 2),
            "Kurtosis": round(valid.kurtosis(), 2),
            "IQR": round(iqr, 2),
            "Outliers": len(outliers),
            "Outlier %": round((len(outliers) / len(valid)) * 100, 2),
        }
    )

print(pd.DataFrame(stat_summary).to_string(index=False))

# 4. Correlation Matrix
print("\n================ CORRELATION MATRIX (COMPLETED RIDES) ================")
corr_cols = [
    "booking_value",
    "ride_distance",
    "fare_per_km",
    "avg_vtat",
    "avg_ctat",
    "customer_rating",
]
completed_df = df[df["booking_status"] == "Completed"][corr_cols]
corr_matrix = completed_df.corr(method="pearson").round(3)
print(corr_matrix)

--> [Step 1] Loading cleaned dataset...
--> [Step 2] Engineering features...

================ STATISTICAL PROFILING ================
         Metric   Mean  Median  Std Dev  Skewness  Kurtosis    IQR  Outliers  Outlier %
  booking_value 508.30  414.00   395.81      2.29      9.89 455.00      3435       3.37
  ride_distance  24.64   23.72    14.00      0.13     -1.21  24.36         0       0.00
       avg_vtat   8.46    8.30     3.77      0.31     -0.60   6.00         0       0.00
       avg_ctat  29.15   28.80     8.90      0.05     -1.12  15.20         0       0.00
customer_rating   4.40    4.50     0.44     -0.89      0.65   0.60      3257       3.50

================ CORRELATION MATRIX (COMPLETED RIDES) ================
                 booking_value  ride_distance  fare_per_km  avg_vtat  \
booking_value            1.000          0.006        0.482     0.003   
ride_distance            0.006          1.000       -0.497     0.004   
fare_per_km              0.482         -0.497     